# Pathway Enrichment Analysis

Differentially expressed genes (DEGs) were subjected to pathway enrichment analysis using the GO, KEGG, and Reactome databases to identify potential biological processes, molecular functions, cellular components, and signaling pathways. All analyses were performed using OmicVerse (version 1.7.8).


In [ ]:
import anndata
import scanpy as sc
import scvelo as scv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from adjustText import adjust_text
import adjustText
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
# import pertpy as pt
import omicverse as ov
import re
from pathlib import Path
import os
from datetime import date
from scipy.sparse import issparse
ov.plot_set()

In [ ]:
# Global matplotlib settings for publication-quality figures
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.sans-serif"] = ["Arial"]
mpl.rcParams["pdf.fonttype"] = 42    # Embed TrueType fonts for editable PDF
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["figure.dpi"]  = 300
mpl.rcParams["savefig.dpi"] = 300

In [ ]:
# 创建自定义 colormap
colors = ["#22456F", "#528FAC","#AECAD8","#F2F2F2","#F7D29D","#EEB84B", "#E87A2D"]  # 示例颜色
cmap_custom = LinearSegmentedColormap.from_list("custom_cmap", colors, N=256)


---

### 1. Data preparation

The list of differentially expressed genes (DEGs) and the processed AnnData object were loaded for downstream analyses. To minimize bias introduced by unobserved genes in snRNA-seq, the **background gene universe** was defined as genes **detected in at least one cell** in the QC-filtered dataset (i.e., excluding all-zero genes). 

---



In [ ]:
# Base directory
file_path = r"F:\Code\codex\PlaceCell_snRNAseq\results"

# Define output directory for pathway enrichment figures
out_dir = Path(r"F:\Code\codex\PlaceCell_snRNAseq\results")
out_dir.mkdir(parents=True, exist_ok=True)

# Append date stamp to avoid overwriting previous results
today = date.today().strftime("%Y%m%d")

# Input DEG file
deg_file = os.path.join(file_path, "spatial_candidate_final_genes_20260609.csv")

# Load data
deg_data = pd.read_csv(deg_file, index_col=0)

# Specify cell type
# cell_type = "Astro"
deg_data.head()

In [ ]:
deg_list = deg_data['gene'].tolist()

len(deg_list)

In [ ]:
deg_list_up = list(deg_data[deg_data['label'] == 'Up']['gene'])
len(deg_list_up)
deg_list_down = list(deg_data[deg_data['label'] == 'Down']['gene'])
len(deg_list_down)

In [ ]:
# setting background gene datasets
adata = sc.read_h5ad(r"F:\Data\snRNAseq\PlaceCell\FilteredData\PCAnalysis_all_QC_20260127.h5ad")
adata_annotation = sc.read_h5ad(
    r"F:\Code\codex\PlaceCell_snRNAseq\Data\FilteredData\All_QC_filtered_annotation_20260604.h5ad"
)
assert adata.n_obs == adata_annotation.n_obs
assert (adata.obs_names == adata_annotation.obs_names).all()

# 转移 cell type 注释
adata.obs["cell_types"] = adata_annotation.obs["cell_types"].values

# 去掉 Unassigned
adata = adata[adata.obs["cell_types"] == "CA1-ProS"].copy()

In [ ]:
adata

In [ ]:
# Define background genes as genes detected in at least one cell
adata_bg = adata.copy()
sc.pp.filter_genes(adata_bg, min_cells=1)
bg = adata_bg.var_names.tolist()
len(bg)

In [ ]:
adata

---

### 2. GO enrichment analysis

Gene Ontology (GO) enrichment analysis was performed to interpret the biological functions associated with the differentially expressed genes (DEGs). GO organizes gene annotations into three complementary domains: **Biological Process (BP)**, describing coordinated biological programs; **Molecular Function (MF)**, describing biochemical activities; and **Cellular Component (CC)**, describing subcellular localization.

---


In [ ]:
# Load and prepare GO Biological Process gene sets (Mouse) for enrichment analysis
pathway_dict = ov.utils.geneset_prepare(
    r'F:\Code\Python\snRNAseq\genesets\GO_Biological_Process_2025.txt',
    organism='Mouse'
)
# GO_Molecular_Function_2025
# GO_Biological_Process_2025
# GO_Cellular_Component_2025

In [ ]:
enr_go =ov.bulk.geneset_enrichment(gene_list=deg_list_up,
                                pathways_dict=pathway_dict,
                                pvalue_type='auto',
                                background=bg,
                                organism='mouse',
                                cutoff= 0.25)

In [ ]:
enr_go

In [ ]:
# Keep text as editable text objects in SVG output
plt.rcParams['svg.fonttype'] = 'none'

# Generate gene set enrichment plot
ax = ov.bulk.geneset_plot(
    enr_go,
    figsize=(2, 4),
    fig_title='GO biological process',
    cax_loc=[1.25, 0.45, 0.5, 0.02],
    bbox_to_anchor_used=(0, -15),
    node_diameter=40,
    node_size=[5,10],
    text_knock=7,
    cmap=cmap_custom,
    custom_ticks=[enr_go["logp"].min(), enr_go["logp"].max()] 
)

cbar = ax.collections[0].colorbar
cbar.mappable.set_clim(enr_go["logp"].min(), enr_go["logp"].max())

# Move y-axis ticks and labels to the left side
ax.yaxis.set_ticks_position('left')
ax.yaxis.set_label_position('left')
ax.tick_params(axis='y', labelleft=True, labelright=False)

# Retrieve figure handle for SVG export (DPI affects rasterized elements only)
fig = ax.figure


In [ ]:
df = enr_go.copy()
df["score"] = df["logp"]
df = df.sort_values("score", ascending=False)
# df = enr_go.copy()

df["Term_clean"] = df["Term"].str.replace(r"\s*\(GO:\d+\)", "", regex=True)

In [ ]:
import matplotlib.pyplot as plt

df_plot = df.head(30)

fig, ax = plt.subplots(figsize=(12,6), dpi=200)

ax.barh(df_plot["Term_clean"], df_plot["score"])
ax.invert_yaxis()
ax.set_xlabel("Enrichment score")
ax.set_title("activity–associated pathways")

for i, (score, num) in enumerate(zip(df_plot["score"], df_plot["num"])):
    ax.text(score, i, f"  {num}", va="center")

plt.tight_layout()
plt.show()

In [ ]:


# 1️⃣ 复制数据
df = enr_go.copy()

# 2️⃣ 使用 -log10(FDR) 作为横轴
df["score"] = df["logp"]

# 3️⃣ 去掉 GO ID
df["Term_clean"] = df["Term"].str.replace(r"\s*\(GO:\d+\)", "", regex=True)

# 4️⃣ 删除冗余条目（手动指定删除列表）
remove_terms = [
    "Negative Regulation of Protein Localization to Cell Surface",
    "Sterol Homeostasis",
    "Protein Localization to Mitochondrion",
    "Regulation of Extrinsic Apoptotic Signaling Pathway"
]

df = df[~df["Term_clean"].isin(remove_terms)]

# 5️⃣ 指定主文顺序（按 narrative，而不是纯显著性）
ordered_terms = [
    "Receptor-Mediated Endocytosis",
    "Negative Regulation of Protein Localization",
    "Intracellular Signaling Cassette",
    "Regulation of Proteolysis",
    "Positive Regulation of ERAD Pathway",
    "Cholesterol Homeostasis",
    "Negative Regulation of Wnt Signaling Pathway"
]

df_plot = df.set_index("Term_clean").loc[ordered_terms].reset_index()

# 6️⃣ 作图
fig, ax = plt.subplots(figsize=(8,4), dpi=300)

bars = ax.barh(df_plot["Term_clean"], df_plot["score"])

ax.invert_yaxis()
ax.set_xlabel(r"$-log_{10}$(FDR)")
ax.set_title("GO biological processes associated with spatial selectivity")

# 7️⃣ 在条尾标 gene 数
for i, (score, n) in enumerate(zip(df_plot["score"], df_plot["num"])):
    ax.text(score, i, f"  n={n}", va="center", fontsize=9)

# 8️⃣ 去掉多余边框（主文风格）
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 参数区：你只改这里
# =========================

MODE = "activity"   # "spatial" 或 "activity"

# 手动删除：spatial 版（你原来的）
remove_terms_spatial = [
    "Negative Regulation of Protein Localization to Cell Surface",
    "Sterol Homeostasis",
    "Protein Localization to Mitochondrion",
    "Regulation of Extrinsic Apoptotic Signaling Pathway",
]

# 手动删除：activity 版（建议先用这个保守清理）
remove_terms_activity = [
    # 常见“伪标签/非神经语义”
    "Positive Regulation by Host of Viral Transcription",
    "Modulation by Host of Viral Process",
    "Modulation by Host of Viral Genome Replication",
    "Epstein-Barr Virus Infection",  # 若是 KEGG 可忽略
    "Positive Regulation of Cardiac Muscle Hypertrophy",
    "Positive Regulation of Muscle Hypertrophy",
]

# 冗余组：每组只保留 score 最大者（logp 最大）
redundant_groups_activity = [
    ["Autophagosome Assembly", "Autophagosome Organization"],
    ["Spindle Assembly Checkpoint Signaling", "Mitotic Spindle Assembly Checkpoint Signaling"],
    ["Mitochondrial Membrane Organization", "Mitochondrial Transport"],
]

# ordered_terms：spatial（你原来的）
ordered_terms_spatial = [
    "Receptor-Mediated Endocytosis",
    "Negative Regulation of Protein Localization",
    "Intracellular Signaling Cassette",
    "Regulation of Proteolysis",
    "Positive Regulation of ERAD Pathway",
    "Cholesterol Homeostasis",
    "Negative Regulation of Wnt Signaling Pathway",
]

# ordered_terms：activity-up（主文推荐 6 条代表）
ordered_terms_activity = [
    "Autophagosome Assembly",  # 或 Autophagosome Organization（自动去冗余会二选一）
    "Organelle Assembly",
    "Mitochondrion Organization",
    "Energy Derivation by Oxidation of Organic Compounds",
    "mRNA Destabilization",
    "Chromatin Organization",
]

# 是否启用“冗余组二选一”
ENABLE_REDUNDANCY_FILTER = True

# =========================
# 主流程
# =========================

# 1) 复制数据
df = enr_go.copy()

# 2) 使用 -log10(FDR) 作为横轴（你这里的 logp）
df["score"] = df["logp"]

# 3) 去掉 GO ID
df["Term_clean"] = df["Term"].str.replace(r"\s*\(GO:\d+\)", "", regex=True)

# 4) 按模式删条目
if MODE == "spatial":
    remove_terms = remove_terms_spatial
    ordered_terms = ordered_terms_spatial
elif MODE == "activity":
    remove_terms = remove_terms_activity
    ordered_terms = ordered_terms_activity
else:
    raise ValueError("MODE must be 'spatial' or 'activity'")

df = df[~df["Term_clean"].isin(remove_terms)].copy()

# 4.5) activity 模式：可选去冗余（每组保留 score 最大）
if MODE == "activity" and ENABLE_REDUNDANCY_FILTER:
    keep = set(df["Term_clean"])
    for group in redundant_groups_activity:
        present = [t for t in group if t in keep]
        if len(present) <= 1:
            continue
        # 保留 score 最大（更显著）
        top = (
            df[df["Term_clean"].isin(present)]
            .sort_values("score", ascending=False)
            .iloc[0]["Term_clean"]
        )
        for t in present:
            if t != top:
                keep.discard(t)
    df = df[df["Term_clean"].isin(keep)].copy()

# 5) 安全索引：缺少条目不会报错（按 ordered_terms 顺序）
df_idx = df.set_index("Term_clean")
terms_present = [t for t in ordered_terms if t in df_idx.index]
df_plot = df_idx.loc[terms_present].reset_index()

# 6) 作图
fig, ax = plt.subplots(figsize=(8, 4), dpi=300)
ax.barh(df_plot["Term_clean"], df_plot["score"])

ax.invert_yaxis()
ax.set_xlabel(r"$-log_{10}$(FDR)")

title = (
    "GO biological processes associated with spatial selectivity"
    if MODE == "spatial"
    else "GO biological processes associated with neuronal activity"
)
ax.set_title(title)

# 7) 条尾标 gene 数
for i, (score, n) in enumerate(zip(df_plot["score"], df_plot["num"])):
    ax.text(score, i, f"  n={n}", va="center", fontsize=9)

# 8) 主文风格
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# 9) 右侧留白，避免 n= 贴边
if len(df_plot) > 0:
    ax.set_xlim(0, df_plot["score"].max() * 1.12)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import re

# 复制原始数据
df_clean = df.copy()

# 1️⃣ 删除泛化条目
generic_keywords = [
    "organelle",
    "intracellular",
    "nucleus",
    "nuclear",
    "nucleolus",
    "external side",
]

pattern_generic = re.compile("|".join(generic_keywords), flags=re.IGNORECASE)

df_clean = df_clean[~df_clean["Term_clean"].str.contains(pattern_generic)]

In [ ]:
# 2️⃣ 定义语义 cluster
cluster_dict = {
    "microtubule": ["microtubule"],
    "intermediate_filament": ["intermediate filament"],
    "lysosome": ["lysosome", "lytic vacuole"],
    "vesicle_transport": ["vesicle", "ER", "COPII"],
    "synapse": ["synapse"],
}

selected_rows = []

for cluster, keywords in cluster_dict.items():
    pattern = re.compile("|".join(keywords), flags=re.IGNORECASE)
    sub = df_clean[df_clean["Term_clean"].str.contains(pattern)]
    
    if len(sub) > 0:
        # 取该cluster中score最高的
        selected_rows.append(sub.sort_values("score", ascending=False).iloc[0])

df_cluster = pd.DataFrame(selected_rows)

In [ ]:
# 已选 term
selected_terms = df_cluster["Term_clean"].tolist()

# 剩余条目中按score排序补足到最多12个
df_remaining = df_clean[~df_clean["Term_clean"].isin(selected_terms)]
df_remaining = df_remaining.sort_values("score", ascending=False)

df_final = pd.concat([
    df_cluster,
    df_remaining.head(12 - len(df_cluster))
]).sort_values("score", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9,5), dpi=300)

ax.barh(df_final["Term_clean"], df_final["score"])
ax.invert_yaxis()

ax.set_xlabel("$-log_{10}$(FDR)")
ax.set_title("GO cellular component associated with spatial selectivity")

for i, (score, num) in enumerate(zip(df_final["score"], df_final["num"])):
    ax.text(score, i, f"  n={num}", va="center", fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def map_system(term):
    t = term.lower()

    # A RNA granules
    if ("membraneless" in t) or ("stress granule" in t):
        return "RNA regulatory granules"

    # D Autophagy–lysosomal/vacuolar
    if ("autophag" in t) or ("vacuolar" in t) or ("v-type atpase" in t) or ("proton-transporting v-type atpase" in t):
        return "Autophagy–lysosomal / vacuolar system"

    # C Mitochondria
    if "mitochond" in t:
        return "Mitochondrial architecture & metabolism"

    # E Nucleus & chromatin remodelers
    if ("nucleus" in t) or ("nuclear" in t) or ("nucleolus" in t) or ("baf" in t) or ("sam complex" in t) or ("mib complex" in t):
        return "Nuclear compartment & chromatin remodeling"

    # F Endomembrane trafficking
    if ("endoplasmic reticulum" in t) or ("golgi" in t) or ("flotillin" in t):
        return "Endomembrane trafficking (ER–Golgi–rafts)"

    # G Cell cycle / spindle
    if ("mitotic spindle" in t) or (t.strip() == "spindle"):
        return "Cell cycle / spindle"

    # B Cytoskeleton/transport/neuron compartment
    if ("dynein" in t) or ("cytoskeleton" in t) or ("spectrin" in t) or ("dendrite" in t) or ("neuron projection" in t) or ("axon" in t) or ("synap" in t):
        return "Cytoskeletal transport & neuronal compartment"

    # 最后兜底：如果还有漏网 term，把它打印出来方便你补规则
    return "UNMAPPED"

df["System"] = df["Term_clean"].apply(map_system)
df = df[~df["Term_clean"].str.contains(
    "intracellular organelle|membrane-bounded organelle|organelle",
    case=False
)]


In [ ]:
df_grouped = (
    df.groupby("System", as_index=False)
      .agg({
          "score": "max",
          "num": "sum"
      })
      .sort_values("score", ascending=False)
)

In [ ]:
import matplotlib.pyplot as plt

df_plot = df_grouped[
    ~df_grouped["System"].isin(["UNMAPPED", "Cell cycle / spindle"])
]

fig, ax = plt.subplots(figsize=(10,5), dpi=300)

ax.barh(df_plot["System"], df_plot["score"])
ax.invert_yaxis()

ax.set_xlabel("$-log_{10}$(FDR)")
ax.set_title("GO cellular component associated with neuronal activity")

for i, (score, num) in enumerate(zip(df_plot["score"], df_plot["num"])):
    ax.text(score, i, f"  n={num}", va="center")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import re

# 复制原始数据
df_clean = df.copy()

# 1️⃣ 删除泛化条目
generic_keywords = [
    "organelle",
    "intracellular",
    "nucleus",
    "nuclear",
    "nucleolus",
    "external side",
]

pattern_generic = re.compile("|".join(generic_keywords), flags=re.IGNORECASE)

df_clean = df_clean[~df_clean["Term_clean"].str.contains(pattern_generic)]

In [ ]:

# Prefer the figure associated with Axes if available
fig = ax.figure if 'ax' in globals() else plt.gcf()

# Save figure as SVG (vector format, publication-ready)
fig.savefig(
    out_dir / f"geneset_GO_Biological_Process_{today}.png",
    format="png",
    dpi=300,                 # DPI applies only to rasterized elements
    bbox_inches="tight"      # Remove extra whitespace
)

print(f"Saved to: {out_dir}")


In [ ]:
# GO Biological Process
pathway_dict_go_bp = ov.utils.geneset_prepare(
    r'F:\Code\Python\snRNAseq\genesets\GO_Biological_Process_2025.txt',
    organism='Mouse'
)
enr_go_bp = ov.bulk.geneset_enrichment(
    gene_list=deg_list,
    pathways_dict=pathway_dict_go_bp,
    pvalue_type='auto',
    background=bg,
    organism='mouse',
    cutoff=0.25
)

# GO Molecular Function
pathway_dict_go_mf = ov.utils.geneset_prepare(
    r'F:\Code\Python\snRNAseq\genesets\GO_Molecular_Function_2025.txt',
    organism='Mouse'
)
enr_go_mf = ov.bulk.geneset_enrichment(
    gene_list=deg_list,
    pathways_dict=pathway_dict_go_mf,
    pvalue_type='auto',
    background=bg,
    organism='mouse',
    cutoff=0.25
)

# GO Cellular Component
pathway_dict_go_cc = ov.utils.geneset_prepare(
    r'F:\Code\Python\snRNAseq\genesets\GO_Cellular_Component_2025.txt',
    organism='Mouse'
)
enr_go_cc = ov.bulk.geneset_enrichment(
    gene_list=deg_list,
    pathways_dict=pathway_dict_go_cc,
    pvalue_type='auto',
    background=bg,
    organism='mouse',
    cutoff=0.25
)


In [ ]:
enr_dict = {
    'BP': enr_go_bp,
    'MF': enr_go_mf,
    'CC': enr_go_cc
}

colors_dict = {
    'BP': ov.pl.red_color[1],
    'MF': ov.pl.green_color[1],
    'CC': ov.pl.blue_color[1],
}

ov.bulk.geneset_plot_multi(
    enr_dict,
    colors_dict,
    num=10,
    figsize=(8, 16),
    text_knock=20,      # 👈 更平衡
    fontsize=15,
    cmap=cmap_custom
)

fig_go = plt.gcf()

fig_go.savefig(
    out_dir / f"geneset_GO_BP_MF_CC_{today}.png",
    format="png",
    dpi=300,                 # DPI applies only to rasterized elements
    bbox_inches="tight"      # Remove extra whitespace
)

print(f"Saved to: {out_dir}")

---

### 3. KEGG Enrichment Analysis

Kyoto Encyclopedia of Genes and Genomes (KEGG) enrichment analysis was performed to identify **signaling and metabolic** pathways significantly associated with the differentially expressed genes (DEGs). KEGG provides manually curated pathway maps that integrate molecular interaction, reaction, and relation networks, enabling functional interpretation at the **pathway and system level**, complementary to Gene Ontology analysis.

---


In [ ]:
# Load and prepare GO Biological Process gene sets (Mouse) for enrichment analysis
pathway_dict_kegg = ov.utils.geneset_prepare(
    r'F:\Code\Python\snRNAseq\genesets\KEGG_2019_Mouse.txt',
    organism='Mouse'
)

In [ ]:
enr_kegg = ov.bulk.geneset_enrichment(
    gene_list=deg_list_up,
    pathways_dict=pathway_dict_kegg,
    organism='mouse',
    pvalue_threshold=0.05,
    background=bg,
    cutoff=0.25
)

In [ ]:
enr_kegg.head()

In [ ]:
enr_kegg["logp"].min(), enr_kegg["logp"].max()

In [ ]:
# Keep text as editable text objects in SVG output
plt.rcParams['svg.fonttype'] = 'none'

# Generate gene set enrichment plot
ax = ov.bulk.geneset_plot(
    enr_kegg,
    figsize=(2, 4),
    fig_title='KEGG',
    cax_loc=[1.25, 0.45, 0.5, 0.02],
    bbox_to_anchor_used=(0, -15),
    node_diameter=40,
    node_size=[5,10],
    text_knock=7,
    cmap=cmap_custom,
    custom_ticks=[enr_kegg["logp"].min(), enr_kegg["logp"].max()] 
)

cbar = ax.collections[0].colorbar
cbar.mappable.set_clim(enr_kegg["logp"].min(), enr_kegg["logp"].max())

# Move y-axis ticks and labels to the left side
ax.yaxis.set_ticks_position('left')
ax.yaxis.set_label_position('left')
ax.tick_params(axis='y', labelleft=True, labelright=False)

# Retrieve figure handle for SVG export (DPI affects rasterized elements only)
fig_kegg = plt.gcf()

fig_kegg.savefig(
    out_dir / f"geneset_KEGG_{today}.png",
    format="png",
    dpi=300,                 # DPI applies only to rasterized elements
    bbox_inches="tight"      # Remove extra whitespace
)

print(f"Saved to: {out_dir}")


---

### 4. Reactome Enrichment Analysis

Reactome enrichment analysis was conducted to further characterize the biological pathways associated with the differentially expressed genes (DEGs) using a **mechanistically curated pathway database**. Unlike GO and KEGG, Reactome organizes pathways in a hierarchical and reaction-centric manner, emphasizing **molecular events, signaling cascades, and biochemical reactions**, thereby providing higher-resolution insight into pathway structure and directionality.

---


In [ ]:
# Load and prepare GO Biological Process gene sets (Mouse) for enrichment analysis
pathway_dict_reactome = ov.utils.geneset_prepare(
    r'F:\Code\Python\snRNAseq\genesets\Reactome_Pathways_2024.txt',
    organism='Mouse'
)

In [ ]:
enr_reactome = ov.bulk.geneset_enrichment(
    gene_list=deg_list_up,
    pathways_dict=pathway_dict_reactome,
    organism='mouse',
    pvalue_threshold=0.05,
    background=bg,
    cutoff=0.25
)

In [ ]:
enr_reactome.head()

In [ ]:
enr_reactome["logp"].min(), enr_reactome["logp"].max()

In [ ]:
# Keep text as editable text objects in SVG output
plt.rcParams['svg.fonttype'] = 'none'

# Generate gene set enrichment plot
ax = ov.bulk.geneset_plot(
    enr_reactome,
    figsize=(2, 4),
    fig_title='Reactome',
    cax_loc=[1.25, 0.45, 0.5, 0.02],
    bbox_to_anchor_used=(0, -15),
    node_diameter=40,
    node_size=[5,10],
    text_knock=7,
    cmap=cmap_custom,
    custom_ticks=[enr_reactome["logp"].min(), enr_reactome["logp"].max()] 
)

cbar = ax.collections[0].colorbar
cbar.mappable.set_clim(enr_reactome["logp"].min(), enr_reactome["logp"].max())

# Move y-axis ticks and labels to the left side
ax.yaxis.set_ticks_position('left')
ax.yaxis.set_label_position('left')
ax.tick_params(axis='y', labelleft=True, labelright=False)

# Retrieve figure handle for SVG export (DPI affects rasterized elements only)
fig_kegg = plt.gcf()

fig_kegg.savefig(
    out_dir / f"geneset_Reactome_{today}.png",
    format="png",
    dpi=300,                 # DPI applies only to rasterized elements
    bbox_inches="tight"      # Remove extra whitespace
)

print(f"Saved to: {out_dir}")
